# Red Team Evaluation - WebSocket Target

Run red team attack techniques against a target reached over a persistent **WebSocket** connection (`ws://` / `wss://`) — e.g. a streaming chat backend or an agent that holds a live session socket.

**How it works:**
1. Each attack session opens (and reuses) one WebSocket connection, keyed by `session_id`, so multi-turn conversations stay on the same socket.
2. For each turn, the handler sends the attack prompt as a message frame and awaits the target's reply frame.

This notebook is **self-contained**: it includes an in-process mock WebSocket target so it runs end-to-end. Point `TARGET_WS_URL` at your real target and adapt `to_target_message` / `from_target_response` to its frame schema.

> Note: this is a **generic WebSocket** target, not a framework's internal socket. For a Streamlit/Gradio chat UI, use `red_team_playwright.ipynb` instead.

**Prerequisites:**
- `pip install -r ../requirements.txt` (includes `aiohttp` for the WebSocket client)
- `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` in `../.env`

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
import asyncio
import json
import os

import aiohttp
from aiohttp import web
from dotenv import load_dotenv
from hiddenlayer import AsyncHiddenLayer

load_dotenv("../.env")

hl_client = AsyncHiddenLayer(
    client_id=os.getenv("HIDDENLAYER_CLIENT_ID"),
    client_secret=os.getenv("HIDDENLAYER_CLIENT_SECRET"),
)

## Configurables

1. Configure target
2. Configure evaluation

In [ ]:
# Target configuration
TARGET_WS_URL = "ws://localhost:8765/ws"  # your target's WebSocket endpoint
TARGET_MODEL_NAME = "websocket-agent"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."
CUSTOM_HEADERS = {
    # "Authorization": "Bearer your-api-key",
}
RECV_TIMEOUT = 120  # seconds to wait for a reply frame per turn

# Mock target server (self-contained demo)
MOCK_HOST = "0.0.0.0"
MOCK_PORT = 8765

# Evaluation configuration
EVAL_NAME = "WebSocket Red Team Eval"  # Name of the evaluation
MAX_TURNS = 5  # Options: 1-5
SESSIONS_PER_TECHNIQUE = 1  # Options: 1-5
PARALLEL_TECHNIQUES = 5  # Options: 1-10
HIDDENLAYER_PROJECT_ID = None  # Options: None, <project_id>

## Verify payload adapters

- `to_target_message` serializes the attack prompt + history into the frame your target expects
- `from_target_response` extracts the reply text from the target's response frame

In [ ]:
def to_target_message(prompt, history, session_id, target_system_prompt):
    """Serialize the attack payload into the target's WebSocket frame schema."""
    messages = [{"role": "system", "content": target_system_prompt or TARGET_SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({"role": "user", "content": prompt})
    return json.dumps({"model": TARGET_MODEL_NAME, "messages": messages})


def from_target_response(data):
    """Extract the reply text from the target's response frame."""
    return data.get("output") or data.get("content") or ""

## Mock WebSocket Target (self-contained)

A tiny in-process `aiohttp` WebSocket server so the notebook runs end-to-end. It echoes a canned refusal for each message and keeps the connection open for multi-turn. Replace `TARGET_WS_URL` with your own endpoint to remove it.

In [ ]:
async def mock_ws_target(request):
    ws = web.WebSocketResponse()
    await ws.prepare(request)
    async for msg in ws:
        if msg.type == aiohttp.WSMsgType.TEXT:
            data = json.loads(msg.data)
            last_user = data["messages"][-1]["content"]
            output = f"[mock ws target] I can't help with that. (received: {last_user[:60]})"
            await ws.send_str(json.dumps({"output": output}))
    return ws


async def start_mock_server():
    """Start the mock WebSocket server in the current event loop; returns the runner for cleanup."""
    app = web.Application()
    app.router.add_get("/ws", mock_ws_target)
    runner = web.AppRunner(app)
    await runner.setup()
    await web.TCPSite(runner, MOCK_HOST, MOCK_PORT).start()
    return runner

## Create Handler

The handler reuses one WebSocket connection per `session_id` (so multi-turn attacks stay on the same socket), sends the attack prompt, and awaits the reply frame.

In [ ]:
# session_id -> (ClientSession, WebSocket)
connections: dict[str, tuple] = {}


async def get_connection(session_id):
    if session_id not in connections:
        cs = aiohttp.ClientSession()
        ws = await cs.ws_connect(TARGET_WS_URL, headers=CUSTOM_HEADERS or None)
        connections[session_id] = (cs, ws)
    return connections[session_id][1]


async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and a WebSocket target."""
    ws = await get_connection(session_id)
    await ws.send_str(to_target_message(prompt, history, session_id, target_system_prompt))

    msg = await asyncio.wait_for(ws.receive(), timeout=RECV_TIMEOUT)
    if msg.type == aiohttp.WSMsgType.TEXT:
        return from_target_response(json.loads(msg.data))
    raise RuntimeError(f"Unexpected WebSocket message type: {msg.type}")


async def close_connections():
    """Close all per-session WebSocket connections."""
    for cs, ws in connections.values():
        await ws.close()
        await cs.close()
    connections.clear()

## Run the Evaluation

Start the mock server, open a red team session, and run attack techniques in parallel. Connections and the server are cleaned up in the `finally` block so this cell can be safely re-run.

In [ ]:
runner = await start_mock_server()
print(f"Mock WebSocket target listening at {TARGET_WS_URL}")

try:
    session = await hl_client.evaluation_sessions.red_team.start_session(
        name=EVAL_NAME,
        target_model=TARGET_MODEL_NAME,
        target_system_prompt=TARGET_SYSTEM_PROMPT,
        max_parallel_techniques=PARALLEL_TECHNIQUES,
        sessions_per_technique=SESSIONS_PER_TECHNIQUE,
        max_turns=MAX_TURNS,
        hiddenlayer_project_id=HIDDENLAYER_PROJECT_ID,
    )
    print(f"Session started: {session.workflow_id}")

    await session.run_with_callback_parallel(handler=handler)
    print("Evaluation complete. View transcripts at https://console.hiddenlayer.ai")
finally:
    await close_connections()
    await runner.cleanup()

## Resume a Session

Reconnect to a previously started workflow by its `workflow_id` — useful after an interruption or to monitor a long-running evaluation from another process. `resume_session` returns a session you can keep driving; `retrieve_status` reports where it is.

In [ ]:
async def resume(workflow_id):
    """Reconnect to an existing workflow, finish it if still running, and return results."""
    session = await hl_client.evaluation_sessions.red_team.resume_session(workflow_id=workflow_id)
    status = await hl_client.evaluations.red_team.retrieve_status(workflow_id)
    print(f"Resumed {session.workflow_id} - status: {status.status}")

    if status.status == "RUNNING":
        await session.run_with_callback_parallel(handler=handler)

    return await hl_client.evaluations.red_team.retrieve_evaluation_results(workflow_id)


# results = await resume("<workflow_id>")

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await hl_client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")
print()

print("BY OBJECTIVE")
print("-" * 60)
for obj_id, obj in report["by_objective"].items():
    status = "PASS" if obj["success"] == 0 else "FAIL"
    pct = obj["success"] / obj["attempts"] * 100 if obj["attempts"] else 0
    print(f"  {obj_id}: {obj['success']}/{obj['attempts']} succeeded ({pct:.0f}%) [{status}]")